# Verdeckung interaktiv ausprobieren

Mit den Reglern wählst du Bild, Linie und Störung. Links siehst du das Originalbild mit der Vorhersage,
in der Mitte das gestörte Bild mit alter (weiße ×) und neuer Vorhersage (Punkte), rechts optional die
HiResCAM-Karte: Wohin schaut das Modell für das gestörte Stück?

**Bedingungen:** A = Stück der Ziellinie gestört · B = A + Nachbarlinie ganz gestört ·
C = nur Nachbarlinie · D = A + Kontrollstreifen auf freier Straße

**Füllung:** `inpaint` = Linie wegretuschiert · `median` = mit Straßengrau übermalt ·
`blendung` = heller Lichtfleck wie Sonne/Scheinwerfer

Ausführen: oben *Run All*. Kernel: `.venv` dieses Projekts.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import ipywidgets as w
import numpy as np

from xai_lane import paths
from xai_lane.cam import RowBinTarget, compute_cam
from xai_lane.decode import decode_row_lanes
from xai_lane.experiment import SkipFrame, default_layer, predict, run_frame
from xai_lane.model_loader import load_model
from xai_lane.preprocess import load_gray, to_model_rgb, to_tensor

IMAGES = sorted(paths.TEST_IMAGE_DIR.glob("*.jpg"))
LANE_NAMES = ["links", "Mitte", "rechts"]
LANE_COLORS = ["tab:blue", "tab:green", "tab:red"]
_models = {}

def get_model(name):
    if name not in _models:
        _models[name] = load_model(name)
    return _models[name]

In [ ]:
def zeige(modell, bild, ziel, nachbar, bedingung, fuellung, bereich, cam):
    if ziel == nachbar:
        print("Ziel und Nachbar müssen verschiedene Linien sein.")
        return
    net, cfg = get_model(modell)
    rgb = to_model_rgb(load_gray(IMAGES[bild]), cfg.train_width, cfg.train_height)
    try:
        res = run_frame(net, cfg, rgb, ziel, nachbar, bereich, fuellung, cam_images=())
    except SkipFrame:
        print("Ziel- oder Nachbarlinie wird in diesem Bild nicht erkannt.")
        return
    img, new = res.imgs[bedingung], res.preds[bedingung]
    fig, axes = plt.subplots(1, 3 if cam else 2, figsize=(17 if cam else 11.5, 4.6))
    axes[0].imshow(rgb)
    for lane, p in enumerate(res.lanes0):
        axes[0].plot(p[:, 0], p[:, 1], ".", ms=4, color=LANE_COLORS[lane])
    axes[0].set_title(f"Original: {IMAGES[bild].name[-14:]}")
    axes[1].imshow(img)
    for lane, p in enumerate(new):
        axes[1].plot(res.lanes0[lane][:, 0], res.lanes0[lane][:, 1], "x", ms=3, color="white", alpha=0.7)
        axes[1].plot(p[:, 0], p[:, 1], ".", ms=4, color=LANE_COLORS[lane])
    r = res.table.loc[bedingung]
    axes[1].set_title(f"{bedingung}, {fuellung}: Ziel „{LANE_NAMES[ziel]}“ {r.dx_mittel_px:.1f} px verschoben, "
                      f"P = {r.P_exist_T:.2f}")
    if cam:
        target = RowBinTarget(ziel, res.hidden, np.array(
            [b for b in decode_row_lanes(predict(net, rgb), cfg)[1][res.hidden, ziel]]))
        c = compute_cam(net, to_tensor(img), target, "hirescam", default_layer(cfg))
        axes[2].imshow(img)
        axes[2].imshow(c, cmap="jet", alpha=0.45)
        axes[2].contour(res.masks["T_verdeckt"], levels=[0.5], colors="magenta", linewidths=0.8)
        axes[2].set_title("HiResCAM: wohin schaut das Modell für das gestörte Stück?")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(res.table.round(3).to_string())

In [ ]:
style = {"description_width": "90px"}
layout = w.Layout(width="420px")
ui = dict(
    modell=w.Dropdown(options=list(paths.MODELS), value=paths.DEFAULT_MODEL, description="Modell", style=style),
    bild=w.IntSlider(value=0, min=0, max=len(IMAGES) - 1, step=1, continuous_update=False,
                     description="Bild", style=style, layout=layout),
    ziel=w.Dropdown(options=[(n, i) for i, n in enumerate(LANE_NAMES)], value=1, description="Ziellinie", style=style),
    nachbar=w.Dropdown(options=[(n, i) for i, n in enumerate(LANE_NAMES)], value=2, description="Nachbar", style=style),
    bedingung=w.ToggleButtons(options=["A", "B", "C", "D"], value="A", description="Bedingung", style=style),
    fuellung=w.ToggleButtons(options=["inpaint", "median", "blendung"], value="blendung", description="Füllung", style=style),
    bereich=w.FloatRangeSlider(value=(0.35, 0.65), min=0, max=1, step=0.05, continuous_update=False,
                               description="Bereich", style=style, layout=layout),
    cam=w.Checkbox(value=False, description="HiResCAM anzeigen (langsamer)"),
)
out = w.interactive_output(zeige, ui)
display(w.VBox([w.HBox([ui["modell"], ui["ziel"], ui["nachbar"]]), ui["bild"], ui["bereich"],
                ui["bedingung"], ui["fuellung"], ui["cam"]]), out)